In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
from librosa import display
from scipy import interpolate
import glob
import mne
from eelbrain import *
import os
import pandas as pd

#### Get envelope

In [ ]:
stim, sr = librosa.load('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav', sr=None)
rms = librosa.feature.rms(y=stim)
times = librosa.times_like(rms)

ts = np.arange(0, 600, 1)
print(len(ts))

spl = interpolate.UnivariateSpline(times, rms[0])
interp_signal = spl(ts)

print(len(interp_signal))

e = np.pad(interp_signal, pad_width=(100, (701 - (len(ts)+100))))
e = e.astype('<f8')
e = np.where(np.isfinite(e), e, 0)

print(len(e))

In [ ]:
stim_dict = {
                # '1001': './sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav', 
                '1002': './sounds/deviant_looming_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav', 
                '1003': './sounds/deviant_receding_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav', 
                '1004': './sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'
            }

In [ ]:
def get_rms_envelope(audio_file):

    stim, sr = librosa.load(audio_file)
    rms = librosa.feature.rms(y=stim)
    times = librosa.times_like(rms)

    ts = np.arange(0, 600, 1)

    spl = interpolate.UnivariateSpline(times, rms[0])
    interp_signal = spl(ts)

    e = np.pad(interp_signal, pad_width=(100, (701 - (len(ts)+100))))
    e = e.astype('<f8')
    e = np.where(np.isfinite(e), e, 0)

    return e

#### Create dataset

In [ ]:
subjs_all = glob.glob('./analysis/looming*-epo.fif')
subjs = []
for i in subjs_all:
    subjs.append(i.lower())

subjs = sorted(subjs, key=lambda x: int(x.split('-')[0][-3:]))
print(subjs)

In [ ]:
tstep = 1. / 1000
n_times = 701
time = UTS(0, tstep, n_times)

sensor = Sensor.from_montage('easycap-M1')[:64]

rows = []

event_dict = {
                # '1001': 'Standard',
                '1002': 'Looming', 
                '1003': 'Receding',
                '1004': 'Deviant'
            }
    

for subj in subjs_all:
    subj_epochs = mne.read_epochs(subj, preload=True, verbose=False)
    subj_epochs = subj_epochs.drop_channels('STI')

    for key, val in event_dict.items():
        for epoch in subj_epochs[key].get_data():
            subject = int(subj.split('-')[0][-3:])

            eeg = NDVar(epoch.T, (time, sensor), name='EEG', info={'unit': 'µV'})

            e = get_rms_envelope(stim_dict.get('1004'))
            envelope = NDVar(e, (time,), name='envelope')

            rows.append([subject, eeg, envelope])

ds = Dataset.from_caselist(['subject','eeg', 'envelope'], rows)
ds['subject'].random = True
print(ds.summary())


# for i in subjs:
#     subj = mne.read_epochs(k)
#     subj = subj.drop_channels('STI')

#     for j in uniq_trials:

#         subject = int(k[17:21])

#         eeg = NDVar(subj[str(j)].get_data()[0].T, (time, sensor), name='EEG', info={'unit': 'µV'})

#         e = get_pitch_envelope(sound, sr=1000)
#         envelope = NDVar(e, (time,), name='envelope')
        
#         rows.append([subject, eeg, envelope]) 

# ds = Dataset.from_caselist(['subject', 'eeg', 'envelope'], rows)
# ds['subject'].random = True
# print(ds.summary())

#### Compute TRF

In [ ]:
fit = boosting('eeg', 'envelope', 0, 0.600, basis=0.050, ds=ds, delta=0.01, partitions=3, test=1)

In [ ]:
x = NDVar(get_rms_envelope('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'), (time,))
y_pred = convolve(fit.h_scaled, x)
y = ds['eeg']
# y_pred = NDVar(y_pred.x.T, (time, sensor))

print(y.shape)
print(y_pred.shape)

print(type(y))
print(type(y_pred))

plot_args = dict(ncol=1, frame='t', legend=False, colors='k')
plot.UTS([x, fit.h, y_pred.mean('sensor')], ylabel=['Stimulus (x)', 'TRF', 'Response (y)'], **plot_args);
# plot.UTS(y_pred, '.sensor')